# Turtle Breakout 05 — is the turtle breakout fractal? 1 minute to 1 day

Pre-registration: `discussions/Turtle_Breakout/007_claude_tb05_preregistration.md`. Code: `src/data_engineering/modeling/turtle_fractal.py`.

Original rules (20-candle high entry, max(entry − 2 ATR, 10-candle low) exit, ATR over 20 candles) on 13 candle sizes: 1, 2, 3, 5, 10, 15, 30 min, 1, 2, 4, 8, 12 h, 1 day. BTC/ETH/SOL/XRP perpetuals, 2020-10-15 → end, unit notional per trade, fills at the close of the trigger bar.

- **Fractal?** edge per trade in risk units, $R_{gross}$, real vs random entries with the same exits, at every candle size.
- **Tradeable?** net bp per trade at taker costs (0.05% + 1 bp per side) and the break-even round-trip cost.

Environment: `TB01_DATA_DIR`, `TB05_SYNTHETIC=1`, `TB05_TFS` (comma list, default all), `TB05_NULL_SCALE`.

In [ ]:
import os, sys, json, time, gc, subprocess, tempfile, warnings, math
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_fractal as fr

SYNTHETIC = os.getenv('TB05_SYNTHETIC', '0') == '1'
TFS = [int(x) for x in os.getenv('TB05_TFS', ','.join(map(str, fr.TIMEFRAMES))).split(',')]
SCALE = float(os.getenv('TB05_NULL_SCALE', '1'))
FP = fr.FrParams(); ASSETS = list(tb.ASSETS)
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb05-')) / '05'; FP = fr.FrParams(n_boot=300); SCALE = min(SCALE, 0.1)
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '05'
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
    DATA = Path(os.environ.get('TB01_DATA_DIR', str(base / 'long')))
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_fractal.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
print(json.dumps(fr.registration(FP), indent=1))

## 1. Data: 1-minute and 5-minute bars, funding

In [ ]:
T0 = time.time(); B1, B5, FUND = {}, {}, {}
for i, a in enumerate(ASSETS):
    if SYNTHETIC:
        rng = np.random.default_rng(i); days = 120; n = days * 1440
        sig = 0.6 / math.sqrt(365 * 1440); path = math.log(100) + np.cumsum(rng.normal(0, sig / math.sqrt(6), (n, 6)).ravel()).reshape(n, 6)
        o = np.r_[math.log(100), path[:-1, -1]]
        t = pd.date_range(tb.COMMON_START - pd.Timedelta(days=30) + pd.Timedelta(minutes=1), periods=n, freq='1min')
        b1 = pd.DataFrame({'open': np.exp(o), 'high': np.exp(np.maximum(path.max(1), o)), 'low': np.exp(np.minimum(path.min(1), o)),
                           'close': np.exp(path[:, -1])}, index=pd.DatetimeIndex(t, name='end'))
        f = tb.synthetic_funding(b1, 0.0001)
    else:
        raw = pd.read_parquet(DATA / f'{a}USDT_um_1m.parquet', columns=['open_time', 'open', 'high', 'low', 'close'])
        b1 = fr.bars_1m(raw); del raw
        f = tb.funding_events(pd.read_parquet(DATA / 'basis' / 'funding' / f'{a}USDT_funding.parquet'))
    b1 = b1[b1.index >= tb.COMMON_START - pd.Timedelta(days=30)]
    B1[a] = b1
    B5[a] = b1.resample('5min', label='right', closed='right').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'}).dropna()
    FUND[a] = f
    print(a, len(b1), 'x 1m |', len(B5[a]), 'x 5m |', f'{time.time() - T0:.0f}s')
END = min(b.index[-1] for b in B1.values())
DAYS = (END - tb.COMMON_START).total_seconds() / 86400
print('window', tb.COMMON_START, '->', END, f'({DAYS:.0f} days)')

## 2. Every candle size: real trades, random-entry null, summary

In [ ]:
SUM, BYA, NULLS, TRADES = [], [], {}, {}
for tf in TFS:
    t1 = time.time()
    frames = {a: fr.prepare((B1 if tf < 5 else B5)[a], FUND[a], tf, FP) for a in ASSETS}
    k0 = {a: fr.k_start_for(F, tb.COMMON_START) for a, F in frames.items()}
    parts, pe = [], {}
    for a, F in frames.items():
        T, st = fr.simulate(F, FP, k0[a]); T['asset'] = a; parts.append(T); pe[a] = st['p_entry']
    T = pd.concat(parts, ignore_index=True)
    n_null = max(10, int(round(fr.N_NULL[tf] * SCALE)))
    null = fr.null_means(frames, FP, pe, k0, n_null, FP.seed + tf)
    s = fr.summarize(tf, T, null, FP, DAYS); s['null_runs'] = n_null; SUM.append(s); NULLS[tf] = null
    BYA.append(T.groupby('asset').agg(trades=('R_gross', 'size'), mean_R_gross=('R_gross', 'mean'), mean_bp_gross=('bp_gross', 'mean'),
                                      mean_bp_net=('bp_net', 'mean'), median_hold_min=('hold_min', 'median')).assign(tf_min=tf))
    if tf >= 30: TRADES[tf] = T
    print(f"{tf:5d} min: {len(T):7d} trades | R_gross {s['mean_R_gross']:+.3f} vs null {s['null_mean_R_gross']:+.3f} (z {s['z']:+.1f}, p {s['p_z']:.4f}) "
          f"| gross {s['mean_bp_gross']:+.1f} bp, net {s['mean_bp_net']:+.1f} bp | cost {s['median_cost_R']:.2f} R | {s['verdict']} ({time.time() - t1:.0f}s)")
    del frames; gc.collect()
S = pd.DataFrame(SUM).set_index('tf_min'); BA = pd.concat(BYA).reset_index()
display(S.round(3))

## 3. Figures

In [ ]:
x = S.index.to_numpy(float); lab = [f'{int(t)}m' if t < 60 else (f'{int(t // 60)}h' if t < 1440 else '1d') for t in x]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
lo = [np.nanquantile(NULLS[t], 0.025) for t in S.index]; hi = [np.nanquantile(NULLS[t], 0.975) for t in S.index]
axes[0].fill_between(x, lo, hi, color='0.85', label='random entries, 95%')
axes[0].plot(x, S.null_mean_R_gross, color='0.5', lw=1, label='random entries, median')
axes[0].plot(x, S.mean_R_gross, 'o-', color='#1f5aa6', lw=1.5, label='breakout')
axes[0].axhline(0, color='k', lw=0.6); axes[0].set_ylabel('mean R per trade, before costs'); axes[0].set_title('Is the edge fractal? (risk units)', loc='left', fontsize=10)
axes[1].plot(x, S.breakeven_rt_cost_bp, 'o-', color='#1f5aa6', label='break-even round-trip cost')
axes[1].axhline(2e4 * FP.side_cost, color='#b33f40', ls='--', lw=1, label=f'taker {2e4 * FP.side_cost:.0f} bp')
axes[1].axhline(2e4 * FP.maker_side_cost, color='#3a7d44', ls=':', lw=1, label=f'maker scenario {2e4 * FP.maker_side_cost:.0f} bp')
axes[1].set_yscale('symlog', linthresh=1); axes[1].set_ylabel('bp per trade'); axes[1].set_title('Can it pay its costs? (price units)', loc='left', fontsize=10)
axes[2].plot(x, S.trades_per_day, 'o-', color='#c2571a', label='trades / day (4 coins)'); ax2 = axes[2].twinx()
ax2.plot(x, S.median_hold_min / 60, 's--', color='#8a4fa3', label='median hold (h)'); axes[2].set_yscale('log'); ax2.set_yscale('log')
axes[2].set_ylabel('trades per day'); ax2.set_ylabel('median hold, hours'); axes[2].set_title('Activity', loc='left', fontsize=10)
for ax in axes:
    ax.set_xscale('log'); ax.set_xticks(x); ax.set_xticklabels(lab, rotation=45, fontsize=8); ax.grid(alpha=0.25); ax.legend(frameon=False, fontsize=7, loc='best')
plt.tight_layout(); plt.show()

## 4. Exports

In [ ]:
S.to_csv(OUT / '05_summary.csv', float_format='%.6g'); BA.to_csv(OUT / '05_by_asset.csv', index=False, float_format='%.6g')
pd.DataFrame({tf: pd.Series(v) for tf, v in NULLS.items()}).to_csv(OUT / '05_null_runs.csv', index=False, float_format='%.5g')
if TRADES: pd.concat([v.assign(tf_min=k) for k, v in TRADES.items()], ignore_index=True).to_csv(OUT / '05_trades_30m_plus.csv.gz', index=False, float_format='%.6g')
(OUT / '05_decision.json').write_text(json.dumps({'by_timeframe': S[['trades', 'mean_R_gross', 'edge_R', 'z', 'p_z', 'null_share_ge_real', 'breakeven_rt_cost_bp',
    'mean_bp_net', 'net_lo', 'G_edge', 'G_net', 'verdict']].reset_index().to_dict('records'), 'null_scale': SCALE, 'registration': fr.registration(FP)}, indent=1, default=float))
(OUT / '05_manifest.json').write_text(json.dumps({'module': 'Turtle_Breakout_05', 'version': fr.VERSION, 'synthetic': SYNTHETIC, 'timeframes': TFS,
    'window': [str(tb.COMMON_START), str(END)], 'runtime_s': round(time.time() - T0, 1)}, indent=1))
print(sorted(p.name for p in OUT.iterdir()))